In [ ]:
import json


with open('matches_2425.json', 'r', encoding='utf-8') as f:
    data = json.load(f)

print(type(data)) #checking dict or list?
print(data[0] if isinstance(data, list) else list(data.keys()))

<class 'list'>
{'date': '2024-08-16', 'time': '20:00 (21:00)', 'day': 'Fri', 'home': 'Manchester Utd', 'away': 'Fulham', 'goals_home': 1, 'goals_away': 0, 'attendance': '73,297', 'venue': 'Old Trafford', 'referee': 'Robert Jones', 'details': {'home': {'players': [{'name': 'André Onana', 'played_from': 0, 'played_until': 90, 'yellow_card': None, 'red_card': None, 'goals': [], 'assists': []}, {'name': 'Noussair Mazraoui', 'played_from': 0, 'played_until': 81, 'yellow_card': None, 'red_card': None, 'goals': [], 'assists': []}, {'name': 'Harry Maguire', 'played_from': 0, 'played_until': 81, 'yellow_card': 40, 'red_card': None, 'goals': [], 'assists': []}, {'name': 'Lisandro Martínez', 'played_from': 0, 'played_until': 90, 'yellow_card': None, 'red_card': None, 'goals': [], 'assists': []}, {'name': 'Mason Mount', 'played_from': 0, 'played_until': 61, 'yellow_card': 18, 'red_card': None, 'goals': [], 'assists': []}, {'name': 'Bruno Fernandes', 'played_from': 0, 'played_until': 90, 'yellow_ca

In [4]:
import json
import pandas as pd

with open('matches_2425.json', 'r', encoding='utf-8') as f:
    data = json.load(f)

print(f"Total matches: {len(data)}")

matches = []
player_appearances = []
match_events = []

for match in data:
    # Generate unique match_id
    match_id = f"{match['date']}_{match['home']}_{match['away']}".replace(' ', '_')

    # ── Table 1: fact_matches (one row per match) ──
    matches.append({
        'match_id':     match_id,
        'date':         match['date'],
        'time':         match.get('time'),
        'day':          match.get('day'),
        'home_team':    match['home'],
        'away_team':    match['away'],
        'goals_home':   match['goals_home'],
        'goals_away':   match['goals_away'],
        'attendance':   str(match.get('attendance', '')).replace(',', ''),
        'venue':        match.get('venue'),
        'referee':      match.get('referee'),
        'result':       'H' if match['goals_home'] > match['goals_away']
                        else ('A' if match['goals_home'] < match['goals_away'] else 'D')
    })

    # ── Table 2: fact_player_appearances (one row per player per match) ──
    for side in ['home', 'away']:
        team_name = match[side]
        players = match.get('details', {}).get(side, {}).get('players', [])
        for p in players:
            player_appearances.append({
                'match_id':       match_id,
                'date':           match['date'],
                'team':           team_name,
                'side':           side,
                'player_name':    p['name'],
                'played_from':    p['played_from'],
                'played_until':   p['played_until'],
                'minutes_played': p['played_until'] - p['played_from'],
                'yellow_card':    1 if p['yellow_card'] else 0,
                'red_card':       1 if p['red_card'] else 0,
                'goals':          len(p['goals']),
                'assists':        len(p['assists']),
            })

    # ── Table 3: fact_match_events (one row per event per match) ──
    for side in ['home', 'away']:
        team_name = match[side]
        events = match.get('details', {}).get(side, {}).get('events', [])
        for e in events:
            match_events.append({
                'match_id':   match_id,
                'date':       match['date'],
                'team':       team_name,
                'side':       side,
                'minute':     e['minute'],
                'event_type': e['type'],
                'player':     e['player'],
                'sub_player': e.get('sub_player'),
                'assist':     e.get('assist'),
            })

# Convert to DataFrames
df_matches = pd.DataFrame(matches)
df_players = pd.DataFrame(player_appearances)
df_events  = pd.DataFrame(match_events)

# Verify shapes
print(f"\n fact_matches:            {df_matches.shape}")
print(f" fact_player_appearances: {df_players.shape}")
print(f" fact_match_events:       {df_events.shape}")

# Quick sanity check — top scorers
print(f"\nTop 10 scorers:")
print(df_players.groupby('player_name')['goals'].sum()
      .sort_values(ascending=False).head(10))

# Save as Parquet for GCS upload
df_matches.to_parquet('fact_matches.parquet', index=False)
df_players.to_parquet('fact_player_appearances.parquet', index=False)
df_events.to_parquet('fact_match_events.parquet', index=False)

print("\n Saved 3 Parquet files successfully!")

Total matches: 380

 fact_matches:            (380, 12)
 fact_player_appearances: (11568, 12)
 fact_match_events:       (5960, 9)

Top 10 scorers:
player_name
Mohamed Salah           20
Yoane Wissa             19
Erling Haaland          19
Alexander Isak          19
Chris Wood              17
Bryan Mbeumo            15
Matheus Cunha           15
Ollie Watkins           14
Jørgen Strand Larsen    14
Luis Díaz               13
Name: goals, dtype: int64

 Saved 3 Parquet files successfully!


In [13]:
import os
from google.cloud import storage

os.environ["GOOGLE_APPLICATION_CREDENTIALS"] = "epl-data-pipeline-9999-de6a9c38ce47.json"

client = storage.Client()
print(f"Connected! Project: {client.project}")

Connected! Project: epl-data-pipeline-9999


In [12]:
# Cài đúng package
!pip install google-cloud-storage google-cloud-bigquery

   ---------------------------------------- 0.0/324.5 kB ? eta -:--:--
   --- ------------------------------------ 30.7/324.5 kB 1.3 MB/s eta 0:00:01
   ----------- ---------------------------- 92.2/324.5 kB 1.1 MB/s eta 0:00:01
   ---------------- --------------------- 143.4/324.5 kB 944.1 kB/s eta 0:00:01
   ---------------------- ----------------- 184.3/324.5 kB 1.0 MB/s eta 0:00:01
   ------------------------------- -------- 256.0/324.5 kB 1.0 MB/s eta 0:00:01
   ------------------------------- -------- 256.0/324.5 kB 1.0 MB/s eta 0:00:01
   ------------------------------- -------- 256.0/324.5 kB 1.0 MB/s eta 0:00:01
   -------------------------------------  317.4/324.5 kB 819.2 kB/s eta 0:00:01
   -------------------------------------- 324.5/324.5 kB 803.6 kB/s eta 0:00:00
   ---------------------------------------- 0.0/262.0 kB ? eta -:--:--
   - -------------------------------------- 10.2/262.0 kB ? eta -:--:--
   ---------- ---------------------------- 71.7/262.0 kB 991.0 kB/s 

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
mediapipe 0.10.21 requires numpy<2, but you have numpy 2.4.0 which is incompatible.
mediapipe 0.10.21 requires protobuf<5,>=4.25.3, but you have protobuf 6.33.6 which is incompatible.

[notice] A new release of pip is available: 24.0 -> 26.0.1
[notice] To update, run: C:\Users\25\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


In [15]:
from google.cloud import bigquery
import os

os.environ["GOOGLE_APPLICATION_CREDENTIALS"] = "epl-data-pipeline-9999-de6a9c38ce47.json"

client = bigquery.Client()

def load_parquet_to_bq(parquet_file, table_id):
    job_config = bigquery.LoadJobConfig(
        source_format=bigquery.SourceFormat.PARQUET,
        write_disposition=bigquery.WriteDisposition.WRITE_TRUNCATE,
        autodetect=True,
    )
    with open(parquet_file, "rb") as f:
        job = client.load_table_from_file(f, table_id, job_config=job_config)
    job.result()  # Wait for job to complete
    table = client.get_table(table_id)
    print(f" Loaded {table.num_rows} rows → {table_id}")

# Load 3 tables từ match data JSON
load_parquet_to_bq(
    "fact_matches.parquet",
    "epl-data-pipeline-9999.epl_raw.fact_matches"
)
load_parquet_to_bq(
    "fact_player_appearances.parquet",
    "epl-data-pipeline-9999.epl_raw.fact_player_appearances"
)
load_parquet_to_bq(
    "fact_match_events.parquet",
    "epl-data-pipeline-9999.epl_raw.fact_match_events"
)

 Loaded 380 rows → epl-data-pipeline-9999.epl_raw.fact_matches
 Loaded 11568 rows → epl-data-pipeline-9999.epl_raw.fact_player_appearances
 Loaded 5960 rows → epl-data-pipeline-9999.epl_raw.fact_match_events


In [ ]:
#check direction
import os
import glob

# Check current working directory
print(f"Current directory: {os.getcwd()}")

# List what's in datasets 9 seasons
print("\nContents of 'datasets 9 seasons':")
for item in os.listdir("datasets 9 seasons"):
    print(f"  {item}")

# Try finding xlsx files
files = glob.glob("datasets 9 seasons/club_stats/*.xlsx")
print(f"\nFound {len(files)} files with current path")

# Try with backslash (Windows)
files2 = glob.glob("datasets 9 seasons\\club_stats\\*.xlsx")
print(f"Found {len(files2)} files with backslash path")

Current directory: d:\Documents\DE Zoomcamp 2026 Final Project

Contents of 'datasets 9 seasons':
  club_stats
  league_table
  player_stats_2024_2025_season.csv
  premier_player_info.csv

Found 0 files with current path
Found 0 files with backslash path


In [3]:
import os
import glob

# Check what's inside club_stats
print("Contents of club_stats:")
for f in os.listdir("datasets 9 seasons/club_stats"):
    print(f"  {f}")

# Check what's inside league_table
print("\nContents of league_table:")
for f in os.listdir("datasets 9 seasons/league_table"):
    print(f"  {f}")

Contents of club_stats:
  2016_season_club_stats.csv
  2017_season_club_stats.csv
  2018_season_club_stats.csv
  2019_season_club_stats.csv
  2020_season_club_stats.csv
  2021_season_club_stats.csv
  2022_season_club_stats.csv
  2023_season_club_stats.csv
  2024_season_club_stats.csv

Contents of league_table:
  away
  home
  home_and_away


In [6]:
import pandas as pd
import glob
import os
from google.cloud import bigquery

os.environ["GOOGLE_APPLICATION_CREDENTIALS"] = "epl-data-pipeline-9999-de6a9c38ce47.json"
client = bigquery.Client()
PROJECT = "epl-data-pipeline-9999"

def load_df_to_bq(df, table_id):
    job_config = bigquery.LoadJobConfig(
        write_disposition=bigquery.WriteDisposition.WRITE_TRUNCATE,
        autodetect=True,
    )
    job = client.load_table_from_dataframe(df, table_id, job_config=job_config)
    job.result()
    table = client.get_table(table_id)
    print(f" {table.num_rows} rows → {table_id}")

# ── Table 1: club_stats (9 seasons) ──
files = glob.glob("datasets 9 seasons/club_stats/*.csv")
print(f"Found {len(files)} club_stats files")

dfs = []
for f in files:
    df = pd.read_csv(f)
    season = os.path.basename(f).split("_")[0]  # "2016"
    df["season"] = season
    dfs.append(df)

df_club = pd.concat(dfs, ignore_index=True)
print(f"Club stats: {df_club.shape} | Columns: {df_club.columns.tolist()}")
load_df_to_bq(df_club, f"{PROJECT}.epl_raw.club_stats")

# ── Table 2: league_table home_and_away ──
files_hw = glob.glob("datasets 9 seasons/league_table/home_and_away/**/*.csv", recursive=True)
print(f"\nFound {len(files_hw)} home_and_away files")

dfs_hw = []
for f in files_hw:
    df = pd.read_csv(f)
    parts = f.replace("\\", "/").split("/")
    season_folder = parts[-2]               # "gameweek_2016"
    season = season_folder.split("_")[1]    # "2016"
    filename = parts[-1]                    # "2016_gameweek_1.csv"
    gameweek = filename.replace(".csv","").split("_")[-1]
    df["season"] = season
    df["gameweek"] = int(gameweek)
    dfs_hw.append(df)

df_league = pd.concat(dfs_hw, ignore_index=True)
print(f"League table: {df_league.shape} | Columns: {df_league.columns.tolist()}")
load_df_to_bq(df_league, f"{PROJECT}.epl_raw.league_table")

# ── Table 3: player_stats 2024-25 ──
df_player_stats = pd.read_csv("datasets 9 seasons/player_stats_2024_2025_season.csv")


Found 9 club_stats files
Club stats: (180, 39) | Columns: ['club_name', 'club_url', 'season', 'Games Played', 'Goals', 'Goals Conceded', 'Shots', 'Shots On Target', 'penalties', 'penalties_scored', 'free_kicks_taken', 'free_kicks_scored', 'Hit Woodwork', 'crosses', 'cross_accuracy', 'Interceptions', 'Blocks', 'Clearances', 'Passes', 'long_passes', 'long_pass_accuracy', 'Corners Taken', 'dribble_attempts', 'dribble_accuracy', 'Duels Won', 'Aerial Duels Won', 'Red Cards', 'Yellow Cards', 'Fouls', 'Offsides', 'Own Goals', 'Free Kicks Scored', 'penalties_saved', 'penalty_save_precentage', 'Penalties Taken', 'XG', 'Shots On Target Inside the Box', 'Shots On Target Outside the Box', 'Touches in the Opposition Box']


C:\Users\25\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\google\cloud\bigquery\_pandas_helpers.py:486: FutureWarning: Loading pandas DataFrame into BigQuery will require pandas-gbq package version 0.26.1 or greater in the future. Tried to import pandas-gbq and got: No module named 'pandas_gbq'
  warnings.warn(


 180 rows → epl-data-pipeline-9999.epl_raw.club_stats

Found 342 home_and_away files
League table: (6520, 13) | Columns: ['position', 'badge_url', 'name', 'games_played', 'games_won', 'games_drawn', 'games_lost', 'goals_for', 'goals_against', 'goal_difference', 'points', 'season', 'gameweek']


C:\Users\25\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\google\cloud\bigquery\_pandas_helpers.py:486: FutureWarning: Loading pandas DataFrame into BigQuery will require pandas-gbq package version 0.26.1 or greater in the future. Tried to import pandas-gbq and got: No module named 'pandas_gbq'
  warnings.warn(


 6520 rows → epl-data-pipeline-9999.epl_raw.league_table


In [7]:
# ── Table 3: player_stats 2024-25 ──
df_player_stats = pd.read_csv("datasets 9 seasons/player_stats_2024_2025_season.csv")
print(f"Player stats: {df_player_stats.shape} | Columns: {df_player_stats.columns.tolist()}")
load_df_to_bq(df_player_stats, f"{PROJECT}.epl_raw.player_stats_2425")

# ── Table 4: premier_player_info ──
df_player_info = pd.read_csv("datasets 9 seasons/premier_player_info.csv")
print(f"Player info: {df_player_info.shape} | Columns: {df_player_info.columns.tolist()}")
load_df_to_bq(df_player_info, f"{PROJECT}.epl_raw.player_info")

Player stats: (1116, 47) | Columns: ['player_name', 'Nationality', 'Preferred Foot', 'Date of Birth', 'appearances_', 'sub_appearances', 'XA', 'pass_attempts', 'pass_accuracy', 'long_pass_attempts', 'long_pass_accuracy', 'Minutes Played', 'Duels Won', 'Total Tackles', 'Interceptions', 'Blocks', 'Red Cards', 'Yellow Cards', 'XG', 'Touches in the Opposition Box', 'Aerial Duels Won', 'Assists', 'Shots On Target Inside the Box', 'cross_attempts', 'cross_accuracy', 'dribble_attempts', 'dribble_accuracy', 'Fouls', 'Goals', 'Hit Woodwork', 'Offsides', 'Shots On Target Outside the Box', 'Corners Taken', 'Appearances', 'free_kick_attempts', 'free_kicks_scored', 'Passes', 'Own Goals', 'Penalties Taken', 'Goals Conceded', 'Clean Sheets', 'Saves Made', 'Penalties Faced', 'penalty_attempts', 'penalties_scored', 'penalties_saved', 'penalty_save_precentage']


C:\Users\25\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\google\cloud\bigquery\_pandas_helpers.py:486: FutureWarning: Loading pandas DataFrame into BigQuery will require pandas-gbq package version 0.26.1 or greater in the future. Tried to import pandas-gbq and got: No module named 'pandas_gbq'
  warnings.warn(


 1116 rows → epl-data-pipeline-9999.epl_raw.player_stats_2425
Player info: (1116, 6) | Columns: ['player_image_url', 'player_name', 'player_country', 'player_club', 'player_position', 'player_stats_url']


C:\Users\25\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\google\cloud\bigquery\_pandas_helpers.py:486: FutureWarning: Loading pandas DataFrame into BigQuery will require pandas-gbq package version 0.26.1 or greater in the future. Tried to import pandas-gbq and got: No module named 'pandas_gbq'
  warnings.warn(


 1116 rows → epl-data-pipeline-9999.epl_raw.player_info
